---



# Unit 2 Assignment: Mixture of Experts (MoE) Router

## 1. Objective

Build a **Smart Customer Support Router** that sends user requests to the right expert:
- **Technical Expert**: bugs, coding, debugging
- **Billing Expert**: charges, refunds, invoices
- **General Expert**: fallback for normal chat
- **Tool Expert (Bonus)**: calls a mock tool for real-time-like data (Bitcoin price)

We will use **Groq API** with different **system prompts** to simulate domain experts.

## 2. Architecture (Router + Experts)

```mermaid
graph TD
    U[User Query] --> R[route_prompt(user_input)]
    R -->|technical| T[Technical Expert Prompt]
    R -->|billing| B[Billing Expert Prompt]
    R -->|general| G[General Expert Prompt]
    R -->|tool| X[Tool Expert Function]
    T --> O[Final Response]
    B --> O
    G --> O
    X --> O
```

In [7]:
# Setup
%pip install groq python-dotenv --upgrade --quiet

from dotenv import load_dotenv
load_dotenv()

import os
import getpass
from groq import Groq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

client = Groq(api_key=os.environ["GROQ_API_KEY"])
ROUTER_MODELS = ["llama-3.1-8b-instant", "llama-3.3-70b-versatile"]
EXPERT_MODELS = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant"]

def invoke_with_fallback(messages, temperature, model_candidates):
    last_error = None
    for model_name in model_candidates:
        try:
            completion = client.chat.completions.create(
                model=model_name,
                temperature=temperature,
                messages=messages,
            )
            return completion
        except Exception as e:
            last_error = e
            # Try next model when one is deprecated/decommissioned.
            if "decommissioned" in str(e).lower() or "no longer supported" in str(e).lower():
                continue
            continue

    raise RuntimeError(f"All candidate models failed. Last error: {last_error}")

print("Setup loaded, API client initialized, and fallback model helper ready.")

Note: you may need to restart the kernel to use updated packages.
Setup loaded, API client initialized, and fallback model helper ready.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 3. Define Experts (MODEL_CONFIG)

Each expert uses the same base model but a different **system role**.

In [8]:
MODEL_CONFIG = {
    "technical": {
        "models": EXPERT_MODELS,
        "temperature": 0.7,
        "system_prompt": (
            "You are a Technical Support Expert. Be precise, structured, and code-focused. "
            "When debugging, explain root cause, then provide actionable steps and concise code examples."
        ),
    },
    "billing": {
        "models": EXPERT_MODELS,
        "temperature": 0.7,
        "system_prompt": (
            "You are a Billing Support Expert. Be empathetic, policy-driven, and financially clear. "
            "Ask for relevant transaction details when needed and suggest next steps for refunds/charges."
        ),
    },
    "general": {
        "models": EXPERT_MODELS,
        "temperature": 0.7,
        "system_prompt": (
            "You are a helpful general assistant for customer support. "
            "Be clear, polite, and concise."
        ),
    },
    "tool": {
        "models": ROUTER_MODELS,
        "temperature": 0.0,
        "system_prompt": "You are a tool-routing assistant."
    },
}

print("MODEL_CONFIG created for technical, billing, general, and tool experts.")

MODEL_CONFIG created for technical, billing, general, and tool experts.


## 4. Router Function (Core Task)

Constraint: return only one label from:
`technical`, `billing`, `general`, `tool`

In [9]:
def route_prompt(user_input: str) -> str:
    routing_instruction = (
        "Classify the user input into exactly one category from this list: "
        "[technical, billing, general, tool]. "
        "Use 'tool' only if the user asks for current/live market price data "
        "(example: current price of Bitcoin). "
        "Return ONLY the category name and nothing else."
    )

    completion = invoke_with_fallback(
        messages=[
            {"role": "system", "content": routing_instruction},
            {"role": "user", "content": user_input},
        ],
        temperature=0.0,
        model_candidates=ROUTER_MODELS,
    )

    category = completion.choices[0].message.content.strip().lower()

    # Guardrail: enforce valid output even if model drifts
    valid_categories = {"technical", "billing", "general", "tool"}
    if category not in valid_categories:
        return "general"
    return category

print("route_prompt(user_input) is defined and ready for intent classification.")

route_prompt(user_input) is defined and ready for intent classification.


## 5. Bonus Tool Function

A mock function that simulates a tool call for Bitcoin price queries.

In [10]:
def mock_get_bitcoin_price() -> str:
    # Mocked value for demonstration (replace with real API if needed)
    mock_price_usd = 68421.42
    return f"Current Bitcoin price (mock): ${mock_price_usd:,.2f} USD"

print("mock Bitcoin tool function is defined.")

mock Bitcoin tool function is defined.


## 6. Orchestrator Function

This function runs the full MoE flow:
1. Route user query
2. Pick expert config
3. Call tool or LLM expert
4. Return final answer

In [11]:
def process_request(user_input: str) -> dict:
    category = route_prompt(user_input)

    if category == "tool":
        tool_result = mock_get_bitcoin_price()
        return {
            "category": category,
            "response": tool_result,
        }

    expert = MODEL_CONFIG.get(category, MODEL_CONFIG["general"])

    completion = invoke_with_fallback(
        messages=[
            {"role": "system", "content": expert["system_prompt"]},
            {"role": "user", "content": user_input},
        ],
        temperature=expert["temperature"],
        model_candidates=expert["models"],
    )

    answer = completion.choices[0].message.content.strip()
    return {
        "category": category,
        "response": answer,
    }

print("process_request(user_input) orchestrator is defined.")

process_request(user_input) orchestrator is defined.


## 7. Test Cases

Lets validate routing + expert behavior with assignment-style examples.

In [13]:
tests = [
    "My python script is throwing an IndexError on line 5.",
    "I was charged twice for my subscription this month.",
    "Hey, can you suggest how to stay productive while studying?",
    "What is the current price of Bitcoin right now?",
]

print("executing test cases through the MoE router...")

for q in tests:
    result = process_request(q)
    print("\n" + "=" * 70)
    print(f"User: {q}")
    print(f"Routed To: {result['category']}")
    print(f"Response:\n{result['response']}")

print("all test cases processed.")

executing test cases through the MoE router...

User: My python script is throwing an IndexError on line 5.
Routed To: technical
Response:
### Debugging the IndexError

To debug the `IndexError` on line 5, let's break down the possible root cause and provide actionable steps.

#### Root Cause:
The `IndexError` typically occurs when you try to access an index in a list (or other sequence type) that does not exist. This could be due to:

* Accessing an index that is out of range (e.g., trying to access the 5th element in a list with only 4 elements).
* Attempting to access an element in an empty list.

#### Actionable Steps:

1. **Verify the list is not empty**: Before accessing any element, ensure the list has at least one element.
2. **Check the index value**: Make sure the index value is within the valid range (0 to length of the list - 1).

#### Code Example:

```python
# Example list
my_list = [1, 2, 3, 4]

# Check if the list is not empty
if my_list:
    # Access the first element 

## 8. What You Built

You implemented a working **MoE Router** with:
- Intent classification (`route_prompt`)
- Expert-specific prompting (`MODEL_CONFIG`)
- Orchestration (`process_request`)
- Optional tool routing for live-data style questions

This is a production-style architecture pattern used in modern AI systems.

---

